In [13]:
def solve(n,s):
    ans = [0] * s
    n -= 1
    for i in range(s-1,0,-1):
        v = min(9,n)
        ans[i] = v
        n -=v
    ans[0] = n+1
    return ans
res = solve(15,3)
print(res)

[1, 5, 9]


In [ ]:
# hotel/views.py
from django.db import transaction
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from .models import Hotel

class BookRoomAPIView(APIView):
    def post(self, request, hotel_id):
        try:
            with transaction.atomic():
                # Lock the row so no other transaction can change it until commit
                hotel = Hotel.objects.select_for_update().get(id=hotel_id)

                if hotel.available_rooms <= 0:
                    return Response({"error": "No rooms available"}, status=status.HTTP_400_BAD_REQUEST)

                # Deduct room
                hotel.available_rooms -= 1
                hotel.save()

                # Here, in real microservice setup, you would send booking confirmation to Booking Service
                return Response({"message": "Room booked successfully"}, status=status.HTTP_200_OK)

        except Hotel.DoesNotExist:
            return Response({"error": "Hotel not found"}, status=status.HTTP_404_NOT_FOUND)


In [ ]:
Transactions :-

     with transaction.atomic():

In an application, a transaction is a sequence of operations performed as a single 
logical unit of work that must either complete
entirely or not at all

In [ ]:
Security
*********

1. Use Django’s Built-In Protections
   1. CSRF Protection
   2. SQL Injection Protection
   3. XSS Protection
   4. Clickjacking Protection
   5. Session Security
2. Authentication and Authorization
3. Password Handling
4. HTTPS and Secure Cookies
5. Input Validation and File Upload Security
6. Third-party Packages & Updates
    Regularly update Django and dependencies.
    Avoid installing untrusted third-party packages.

In [ ]:
When an interviewer asks "How is your application scalable?" —
they're testing your understanding of how your app handles increased load (users, data, traffic)
without performance degradation.

1. Vertical and Horizontal Scalability

Vertical Scaling (Scale-Up):
Increase resources (CPU, RAM) of a single server.

Good for early stages, but limited.

Horizontal Scaling (Scale-Out):
******************************
Add more servers/nodes behind a load balancer.

Each instance runs the same application code.

Django works well in a multi-instance setup using tools like Gunicorn + Nginx + AWS ALB.

2. Database Scaling
*******************
🔹 Read Scaling:
Use Read Replicas (PostgreSQL/MySQL) for read-heavy workloads.

Django supports this via database routers.

🔹 Write Scaling:
Optimize schema and queries.

Use sharding or partitioning if needed.

Add caching to reduce DB load.

3. Caching Layer
Use Redis or Memcached to cache:
Frequently accessed views or pages.
Expensive query results.
Session data.
5. Load Balancing
Use NGINX, HAProxy, or cloud-based Load Balancers (AWS ALB, Azure App Gateway).

Distribute traffic evenly across app instances.

4. Asynchronous Task Queues
Offload long-running or heavy tasks (e.g., email, report generation) using:
Celery + Redis/RabbitMQ
Django Q, Huey
8. Auto-Scaling in Cloud
9. Monitoring & Bottleneck Identification

In [ ]:
2.design a csv format report that uses relational data base , 
the data should be list of employee who has left organization and 
joineds the organization for a particular month
	the report will be delived to s3 path every month end,so design a 
solution diagram for this and write a python code for this

In [ ]:
import pandas as pd
import sqlalchemy
import boto3
from datetime import datetime, timedelta

# --- 1. Connect to Database ---
DB_USER = 'your_user'
DB_PASS = 'your_pass'
DB_HOST = 'localhost'
DB_NAME = 'company_db'

engine = sqlalchemy.create_engine(f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}')

# --- 2. Calculate first and last day of current month ---
today = datetime.today()
first_day = today.replace(day=1)
last_day = (first_day + timedelta(days=32)).replace(day=1) - timedelta(days=1)

# --- 3. Fetch employee data ---
query = f"""
SELECT full_name, join_date, exit_date
FROM employees
WHERE (join_date BETWEEN '{first_day.date()}' AND '{last_day.date()}')
   OR (exit_date BETWEEN '{first_day.date()}' AND '{last_day.date()}');
"""

df = pd.read_sql(query, engine)

# --- 4. Save CSV locally ---
csv_file = f"employee_report_{today.strftime('%Y_%m')}.csv"
df.to_csv(csv_file, index=False)

# --- 5. Upload CSV to S3 ---
S3_BUCKET = 'your-bucket-name'
S3_KEY = f'reports/{csv_file}'

s3 = boto3.client('s3')
s3.upload_file(csv_file, S3_BUCKET, S3_KEY)

print(f"Report uploaded to s3://{S3_BUCKET}/{S3_KEY}")


In [ ]:
How do you handle inter-service communication failures

    Retries with exponential backoff
    Circuit Breaker pattern
    Timeouts + Fail-fast strategy
    Fallback / graceful degradation
    Idempotency for retry-safe operations
    Dead-letter queues for async messages
    Centralized logging + tracing (ELK, OpenTelemetry)
    Health checks & service discovery

In [ ]:
from datetime import datetime,timedelta
def get_month_range():
    """Return first and last day of previous month."""
    today = datetime.today()
    first_day_current = today.replace(day=1)
    last_day_prev = first_day_current - timedelta(days=1)
    first_day_prev = last_day_prev.replace(day=1)

    return first_day_prev.date(), last_day_prev.date()

get_month_range()

In [ ]:
from rest_framework import viewsets
from rest_framework.permissions import IsAuthenticatedOrReadOnly
from .models import Book
from .serializers import BookSerializer

router = DefaultRouter()
router.register(r'books', BookViewSet)
urlpatterns = [
    path('api/', include(router.urls)),
]

class BookViewSet(viewsets.ModelViewSet):
    queryset = Book.objects.all()
    serializer_class = BookSerializer
    permission_classes = [IsAuthenticatedOrReadOnly]

In [ ]:
How to Secure Your Python APIs
******************************
 1. ✅ Use HTTPS (TLS/SSL) for all API traffic.
 2. Store API secrets/tokens securely in environment variables.
✅ Add CORS protection using middleware (flask-cors, FastAPI CORSMiddleware).
✅ Implement Rate Limiting 
✅ Use Input Validation to avoid SQL injection/XSS attacks.
✅ Use Role-Based Access Control (RBAC) 

In [ ]:
Built-in DRF Permission Classes:-
  1. AllowAny
  2. IsAuthenticated
  3. IsAdminUser
  4. IsAuthenticatedOrReadOnly	
  5. DjangoModelPermissions
  6. DjangoObjectPermissions

In [ ]:
def solve(str):
    if len(str) == 0:
        return ['']
    res = []
    for i in range(len(str)):
        e = str[:i]+str[i+1:]
        for p in solve(e):
            k = str[i]+p
            res.append(k)
    return res
s = "abc"
res = solve(s)
print(res)

In [ ]:
Context Manager :

A context manager in Python is an object that manages a 
resource and automatically handles setup and cleanup for you.

with open("data.txt", "r") as f:
    data = f.read

🔥 Why use context managers?
Automatic cleanup
Avoid memory leaks

In [ ]:
# REST API Best Practices
1. Use Proper HTTP Methods
2. Use Nouns in Resource URLs (Not Verbs)
3. Use Plural Naming for Collections
4. Support Filtering, Sorting, Pagination
5. Use Consistent and Standardized JSON Responses
6. Version Your API
7. Secure Your API
8. Use Meaningful Error Messages
9. 

In [ ]:
✅ What is namedtuple in Python?

namedtuple is a factory function from the collections module that creates tuple subclasses with named fields.
    
🔹 Why use namedtuple?

Improves readability
Access fields using dot notation
Immutable like tuples
Lightweight compared to classes
Useful for structured data

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
## Best Practise in RestAPI

1. Appropriate Use of HTTP Methods.
2. Stateless Operations
3. Use the Appropriate Request Headers for Authentication
4. Versioning
5. Comprehensive Documentation
6. Error Handling and Status Codes
7. Pagination and Filtering
8.  Use descriptive and meaningful resource names
9. Resource-Based URI Structure
10. Utilize Appropriate HTTP Methods:
11. Implement Consistent Naming Conventions
12. Provide Detailed Error Handling:
13. Maintain Statelessness:
14. 

In [ ]:
List  &  Array

List
****
1. Store different types of data.
2. Dynamic Sizing
3. External Module doesnot Requirement

Array
******
1. Store same types of data  // Homogeneous 
2. Memory Efficiency
3. External Module Requirement

In [ ]:
How to secure api?

1. Implement Strong Authentication and Authorization.
2. Use HTTPS/TLS  : Encrypt data transmitted between clients and your API 
3. Validate and Sanitize Inputs.
4. Regularly Monitor and Log API Activity
5. Keep Software and Dependencies Updated
6. Utilize API Gateways and Firewalls :  filter malicious traffic
7. 


In [ ]:
Rest :-

   . JSON, XML (mostly JSON)
   . Slower (text-based, larger payloads)
   . Widely supported and easy to debug
   . Human readability
   . HTTP status codes
   . Web APIs, public APIs, simple CRUD

grps :-
  . Protocol Buffers (Protobuf - binary)
  . Faster (compact, binary, multiplexed over HTTP/2)
  . Requires code generation from .proto files
  . No readability
  . Microservices communication, high-performance systems, real-time apps


In [ ]:
How to handle millions of concurrent users?

 Ans :-
    
     1. Scalable Architecture
          a. Microservices:-
            Break the system into independent services to scale each
            component individually.
          b. Load Balancing
            Use load balancers (like NGINX, HAProxy, or AWS ELB) 
            to distribute incoming traffic across multiple servers.
          c. Horizontal Scaling
             Add more instances of your application
             
            Use containers (Docker) and orchestration
                tools (Kubernetes) to manage scaling.
     2.  Caching
    
       Use aggressive caching strategies to reduce backend load.
        
        Tools: CDNs (like Cloudflare or Akamai) for static assets 
        and Redis/Memcached for dynamic content.
        
    3, 4. Database Optimization
    
        Sharding: Split data across multiple databases to reduce load on any
        single one.
        Read Replicas: Offload read operations from the primary database.
        Use connection pooling, indexing, and query optimization
    5. Message Queues & Asynchronous Processing
    6. Autoscaling & Cloud Infrastructure
    7. Monitoring and Alerting
    9. Security & DDoS Protection

In [ ]:
3️⃣ Why Async Improves Performance
✅ 1. Better CPU Utilization

CPU doesn’t sit idle waiting for I/O

Handles other requests meanwhile

✅ 2. Higher Throughput

More requests served per second

Especially useful for web APIs

✅ 3. Lower Latency

Requests don’t queue behind slow I/O

Faster response time for users

✅ 4. Scalability

Async apps handle thousands of concurrent users

Fewer threads required → less memory usage